In [4]:
%cd /content
!rm -rf proyecto_IX_data_analyst


/content


In [5]:
!git clone https://github.com/Bootcamp-IA-P5/proyecto_IX_data_analyst.git


Cloning into 'proyecto_IX_data_analyst'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 98 (delta 27), reused 79 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (98/98), 18.64 MiB | 37.73 MiB/s, done.
Resolving deltas: 100% (27/27), done.


In [6]:
%cd proyecto_IX_data_analyst
!git checkout 5-data-cleaning-and-preprocessing || git checkout -b 5-data-cleaning-and-preprocessing


/content/proyecto_IX_data_analyst
Branch '5-data-cleaning-and-preprocessing' set up to track remote branch '5-data-cleaning-and-preprocessing' from 'origin'.
Switched to a new branch '5-data-cleaning-and-preprocessing'


In [7]:
!pwd


/content/proyecto_IX_data_analyst


In [8]:
!mkdir -p data/processed notebooks


In [9]:
!ls


data  LICENSE  notebooks  powerBI  README.md  reports  requirements.txt


# 🧹 02 - Limpieza y preprocesamiento de datos (Madrid)
En este cuaderno, realizamos la limpieza y el preprocesamiento de datos del conjunto de datos de Airbnb para la ciudad de Madrid.

Los pasos incluyen:
1. Cargar el conjunto de datos sin procesar
2. Filtrar los datos de Madrid
3. Gestionar los valores faltantes
4. Eliminar las columnas irrelevantes
5. Codificar y normalizar las variables
6. Gestionar los valores atípicos (con método adaptativo)
7. Guardar el conjunto de datos limpio

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

pd.set_option('display.max_columns', None)
sns.set(style="whitegrid", palette="muted")

print("✅ Libraries successfully imported.")


✅ Libraries successfully imported.


In [11]:
# Cargamos el dataset comprimido (raw)
raw_path = "data/raw/listings.csv.gz"
df = pd.read_csv(raw_path, compression='gzip', low_memory=False)

print(f"✅ Dataset loaded with {df.shape[0]} rows and {df.shape[1]} columns.")
df.head(3)


✅ Dataset loaded with 26004 rows and 79 columns.


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,21853,https://www.airbnb.com/rooms/21853,20250612050748,2025-06-26,city scrape,Bright and airy room,We have a quiet and sunny room with a good vie...,We live in a leafy neighbourhood with plenty o...,https://a0.muscache.com/pictures/68483181/87bc...,83531,https://www.airbnb.com/users/show/83531,Abdel,2010-02-21,"Madrid, Spain",EN-ES-FR\r\nEN\r\nHi everybody: I'm Abdel. I'm...,NaN,NaN,NaN,f,https://a0.muscache.com/im/users/83531/profile...,https://a0.muscache.com/im/users/83531/profile...,Aluche,2.0,2.0,"['email', 'phone']",t,t,"Madrid, Spain",Cármenes,Latina,40.40381,-3.74130,Private room in rental unit,Private room,1,1.0,1 bath,1.0,1.0,"[""Washer"", ""Hair dryer"", ""Coffee maker"", ""Oven...",$29.00,4,40,4.0,4.0,40.0,40.0,4.0,40.0,NaN,t,5,35,65,340,2025-06-26,33,0,0,164,0,0,0.0,2014-10-10,2018-07-15,4.58,4.72,4.56,4.75,4.82,4.21,4.67,NaN,f,2,0,2,0,0.25
1,30320,https://www.airbnb.com/rooms/30320,20250612050748,2025-06-27,previous scrape,Apartamentos Dana Sol,NaN,NaN,https://a0.muscache.com/pictures/336868/f67409...,130907,https://www.airbnb.com/users/show/130907,Dana,2010-05-24,"Madrid, Spain",Apartasol offers a network of several spacious...,NaN,NaN,NaN,f,https://a0.muscache.com/im/users/130907/profil...,https://a0.muscache.com/im/users/130907/profil...,Sol,3.0,6.0,"['email', 'phone']",t,f,NaN,Sol,Centro,40.41476,-3.70418,Entire rental unit,Entire home/apt,2,NaN,1 bath,1.0,NaN,"[""TV with standard cable"", ""Air conditioning"",...",NaN,5,50,5.0,7.0,50.0,50.0,5.1,50.0,NaN,t,2,32,62,337,2025-06-27,172,0,0,160,0,0,NaN,2010-07-06,2022-09-26,4.63,4.71,4.88,4.82,4.78,4.90,4.69,NaN,f,3,3,0,0,0.94
2,30959,https://www.airbnb.com/rooms/30959,20250612050748,2025-06-27,previous scrape,Beautiful loft in Madrid Center,Beautiful Loft 60m2 size just in the historica...,NaN,https://a0.muscache.com/pictures/78173471/835e...,132883,https://www.airbnb.com/users/show/132883,Angela,2010-05-26,"Madrid, Spain",Estoy empezando en Airbnb y deseo que mis hués...,NaN,NaN,NaN,f,https://a0.muscache.com/im/users/132883/profil...,https://a0.muscache.com/im/users/132883/profil...,Embajadores,1.0,4.0,"['email', 'phone']",t,f,NaN,Embajadores,Centro,40.41259,-3.70105,Entire loft,Entire home/apt,2,NaN,1 bath,1.0,NaN,"[""Washer"", ""Breakfast"", ""Essentials"", ""Kitchen...",NaN,3,730,3.0,3.0,730.0,730.0,3.0,730.0,NaN,NaN,0,0,0,0,2025-06-27,8,0,0,0,0,0,NaN,2015-05-12,2017-05-30,4.38,4.14,4.38,4.63,4.63,4.88,4.25,NaN,f,1,1,0,0,0.06


In [12]:
print("Columnas principales del dataset:")
print(df.columns.tolist()[:30])  # mostramos solo las primeras 30


Columnas principales del dataset:
['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_name', 'host_since', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed']


In [13]:
irrelevant_cols = [
    'listing_url', 'scrape_id', 'last_scraped', 'source', 'picture_url',
    'host_thumbnail_url', 'host_picture_url', 'calendar_updated',
    'calendar_last_scraped', 'license', 'neighbourhood_group_cleansed'
]

df.drop(columns=[c for c in irrelevant_cols if c in df.columns], inplace=True, errors='ignore')
print(f"✅ Removed irrelevant columns. Remaining: {df.shape[1]}")


✅ Removed irrelevant columns. Remaining: 68


In [14]:
# Porcentaje de nulos
nulls = df.isnull().mean().sort_values(ascending=False)
display(nulls.head(10))

# Eliminamos columnas con más del 40% de nulos
cols_to_drop = nulls[nulls > 0.4].index
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

# Imputaciones simples
for col in ['beds', 'bedrooms']:
    if col in df.columns:
        df[col].fillna(df[col].median(), inplace=True)

if 'bathrooms_text' in df.columns:
    df['bathrooms_text'] = df['bathrooms_text'].fillna("1 bath")

print(f"✅ Null values handled. Remaining columns: {df.shape[1]}")


,0
host_neighbourhood,0.668166
neighborhood_overview,0.570912
neighbourhood,0.570912
host_about,0.497808
host_location,0.310298
bathrooms,0.228388
beds,0.228234
price,0.227811
estimated_revenue_l365d,0.227811
review_scores_value,0.201738


✅ Null values handled. Remaining columns: 64


/tmp/ipython-input-1770362968.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)


In [15]:
if 'price' in df.columns:
    df['price'] = (
        df['price']
        .astype(str)
        .str.replace(r'[$,]', '', regex=True)
        .astype(float)
    )

print("✅ Converted 'price' column to numeric format.")


✅ Converted 'price' column to numeric format.


### 🧩 Decisión sobre tratamiento de outliers

Se optó por un **método de eliminación suavizado** de outliers, aplicando percentiles **0.01 y 0.99**
dentro de cada barrio (`neighbourhood_group_cleansed`), en lugar de los percentiles más agresivos (0.05–0.95).

**Motivación:**
- Mantener datos representativos de zonas con precios naturalmente altos o bajos (p. ej. Salamanca vs Usera).
- Evitar eliminar propiedades válidas que aportan información útil al modelo.
- Mejorar la generalización reduciendo el sesgo hacia valores medios.

**Resultado:**
La limpieza eliminó un número moderado de registros extremos (outliers reales),
manteniendo la integridad y diversidad del dataset.


In [16]:
# ====================================================
# 🧩 Celda 10 — Outlier handling (soft version, fixed)
# ====================================================

def remove_outliers_by_group(df, group_col, target_col):
    """
    Elimina valores atípicos del target_col dentro de cada grupo del group_col
    utilizando percentiles 0.01 y 0.99 para mantener más datos válidos.
    """
    def iqr_filter(group):
        Q1 = group[target_col].quantile(0.01)
        Q3 = group[target_col].quantile(0.99)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        return group[(group[target_col] >= lower) & (group[target_col] <= upper)]

    filtered_df = df.groupby(group_col, group_keys=False).apply(iqr_filter)
    return filtered_df

# Aplicamos el método a nuestro dataset
before = len(df)
df = remove_outliers_by_group(df, 'neighbourhood_cleansed', 'price')
after = len(df)

print(f"✅ Adaptive outlier removal (soft version) completed. Removed {before - after} rows.")


✅ Adaptive outlier removal (soft version) completed. Removed 5961 rows.


/tmp/ipython-input-3658864239.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  filtered_df = df.groupby(group_col, group_keys=False).apply(iqr_filter)


In [17]:
# ====================================================
# 🧩 Celda 11 — Codificación de variables categóricas
# ====================================================

cat_cols = ['property_type', 'room_type', 'neighbourhood_cleansed']
cat_cols = [c for c in cat_cols if c in df.columns]

df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
print("✅ Applied One-Hot Encoding to categorical variables.")


✅ Applied One-Hot Encoding to categorical variables.


In [18]:
# ====================================================
# 🧩 Celda 12 — Normalización de variables numéricas
# ====================================================

from sklearn.preprocessing import MinMaxScaler

num_cols = ['price', 'minimum_nights', 'maximum_nights', 'number_of_reviews']
num_cols = [c for c in num_cols if c in df.columns]

scaler = MinMaxScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

print("✅ Numerical columns normalized using MinMaxScaler.")


✅ Numerical columns normalized using MinMaxScaler.


In [19]:
# ====================================================
# 🧩 Celda 13 — Guardar dataset limpio procesado
# ====================================================

import os

# Crear la carpeta processed si no existe
os.makedirs("data/processed", exist_ok=True)

# Guardar dataset limpio comprimido
output_path = "data/processed/listings_clean_madrid.csv.gz"
df.to_csv(output_path, index=False, compression="gzip")

print(f"✅ Dataset limpio guardado correctamente en: {output_path}")
print(f"📊 Filas finales del dataset: {len(df)}")


✅ Dataset limpio guardado correctamente en: data/processed/listings_clean_madrid.csv.gz
📊 Filas finales del dataset: 20043


In [20]:
# ====================================================
# 🧩 Celda 14 — Validación final del dataset
# ====================================================

print("📋 Resumen del dataset limpio:")
display(df.describe(include="all").transpose().head(10))

print("\n🧩 Columnas finales:")
print(df.columns.tolist())


📋 Resumen del dataset limpio:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,20043.0,NaN,NaN,NaN,791073857786597760.0,534034607216058816.0,21853.0,51589828.0,961609295406831744.0,1243066174108558336.0,1441028226979579648.0
name,20043,18620,Habitación en piso compartido,172,NaN,NaN,NaN,NaN,NaN,NaN,NaN
description,19482,15892,Our mission is to empower individuals to immer...,207,NaN,NaN,NaN,NaN,NaN,NaN,NaN
host_id,20043.0,NaN,NaN,NaN,290102743.771392,219776307.893962,31622.0,65656674.0,291253690.0,481066735.0,700514774.0
host_url,20043,7782,https://www.airbnb.com/users/show/377605855,329,NaN,NaN,NaN,NaN,NaN,NaN,NaN
host_name,20032,3160,Francisco Andres,490,NaN,NaN,NaN,NaN,NaN,NaN,NaN
host_since,20031,3769,2020-11-30,330,NaN,NaN,NaN,NaN,NaN,NaN,NaN
host_location,13571,297,"Madrid, Spain",11893,NaN,NaN,NaN,NaN,NaN,NaN,NaN
host_response_time,19093,4,within an hour,13830,NaN,NaN,NaN,NaN,NaN,NaN,NaN
host_response_rate,19093,83,100%,11968,NaN,NaN,NaN,NaN,NaN,NaN,NaN



🧩 Columnas finales:
['id', 'name', 'description', 'host_id', 'host_url', 'host_name', 'host_since', 'host_location', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'latitude', 'longitude', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'minimum_nights', 'maximum_nights', 'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'has_availability', 'availability_30', 'availability_60', 'availability_90', 'availability_365', 'number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d', 'availability_eoy', 'number_of_reviews_ly', 'estimated_occupancy_l365d', 'estimated_revenue_l365d', 'first_review', 'last_review', 'review_scores_rating', 'review_scores_accuracy', 'review_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
!find /content/drive -maxdepth 4 -type f -name "02_data_preprocessing.ipynb" -print


find: ‘/content/drive’: No such file or directory


In [ ]:
# Copiamos desde Drive al repo (si existe en ambas rutas, se sobrescribe la más actual)
!cp "/content/drive/MyDrive/02_data_preprocessing.ipynb" "/content/proyecto_IX_data_analyst/notebooks/02_data_preprocessing.ipynb" || cp "/content/drive/MyDrive/Colab Notebooks/02_data_preprocessing.ipynb" "/content/proyecto_IX_data_analyst/notebooks/02_data_preprocessing.ipynb"

# Verificamos que ya está dentro del repo
!ls /content/proyecto_IX_data_analyst/notebooks


01_data_exploration.ipynb  02_data_preprocessing.ipynb	text.txt


In [2]:
# ================================================================
# 💾 Exportar dataset limpio a formato CSV (compatible con Power BI)
# ================================================================

import pandas as pd

# Ruta del archivo comprimido (ajústala si está en otra carpeta)
input_path = "/content/proyecto_IX_data_analyst/data/processed/listings_clean_madrid.csv.gz"

# Leemos el archivo comprimido
df = pd.read_csv(input_path, compression='gzip')

# Nueva ruta de salida sin compresión
output_path = "/content/proyecto_IX_data_analyst/data/processed/listings_clean_madrid.csv"

# Guardamos el archivo en formato CSV normal
df.to_csv(output_path, index=False, encoding='utf-8')

print(f"✅ Archivo convertido correctamente: {output_path}")


FileNotFoundError: [Errno 2] No such file or directory: '/content/proyecto_IX_data_analyst/data/processed/listings_clean_madrid.csv.gz'

In [ ]:
import getpass

# Configuración de usuario
!git config --global user.email "ypimentel.tapia@gmail.com"
!git config --global user.name "Yeder"

# Mensaje de commit (Conventional Commit)
commit_message = "feat(preprocessing): finalize preprocessing with encoding and normalization for Madrid dataset"

# Autenticación
print("🔒 Introduce tu Personal Access Token (no se mostrará al escribir)")
token = getpass.getpass()

# Datos del repo
org_name = "Bootcamp-IA-P5"
repo_name = "proyecto_IX_data_analyst"
branch_name = "5-data-cleaning-and-preprocessing"

# URL segura
push_url = f"https://{token}@github.com/{org_name}/{repo_name}.git"

# Subimos también el dataset limpio junto al notebook
!cd /content/proyecto_IX_data_analyst && git init
!cd /content/proyecto_IX_data_analyst && git remote remove origin || true
!cd /content/proyecto_IX_data_analyst && git remote add origin {push_url}
!cd /content/proyecto_IX_data_analyst && git fetch origin
!cd /content/proyecto_IX_data_analyst && git checkout {branch_name} || git checkout -b {branch_name}

# Añadimos el notebook y el dataset limpio
!cd /content/proyecto_IX_data_analyst && git add notebooks/02_data_preprocessing.ipynb data/processed/

# Commit y push
!cd /content/proyecto_IX_data_analyst && git commit -m "{commit_message}"
!cd /content/proyecto_IX_data_analyst && git push origin {branch_name}

print("\n✅ Notebook y dataset subidos correctamente a GitHub 🎉")


🔒 Introduce tu Personal Access Token (no se mostrará al escribir)
··········
Reinitialized existing Git repository in /content/proyecto_IX_data_analyst/.git/
From https://github.com/Bootcamp-IA-P5/proyecto_IX_data_analyst
 * [new branch]      5-data-cleaning-and-preprocessing -> origin/5-data-cleaning-and-preprocessing
 * [new branch]      copilot/configure-kanban-columns-labels -> origin/copilot/configure-kanban-columns-labels
 * [new branch]      development            -> origin/development
 * [new branch]      feature/choose-dataset -> origin/feature/choose-dataset
 * [new branch]      feature/exploratory-data-analysis -> origin/feature/exploratory-data-analysis
 * [new branch]      main                   -> origin/main
Already on '5-data-cleaning-and-preprocessing'
[5-data-cleaning-and-preprocessing ce11072] feat(preprocessing): finalize preprocessing with encoding and normalization for Madrid dataset
 2 files changed, 1 insertion(+)
 create mode 100644 data/processed/listings_clea

# 🧾 Conclusión del notebook `02_data_preprocessing.ipynb`

En este notebook se realizó todo el proceso de **limpieza y preprocesamiento de los datos** para el dataset de Airbnb en Madrid.  
Las etapas principales fueron:

1. **Carga y exploración inicial del dataset**
   - Revisión de estructura, tipos de datos y valores faltantes.

2. **Limpieza de datos**
   - Eliminación de columnas irrelevantes.
   - Relleno o eliminación de valores nulos según contexto.
   - Conversión de tipos de datos a formatos adecuados (numéricos, fechas, etc.).

3. **Tratamiento de outliers**
   - Se aplicó un método adaptativo basado en percentiles (1–99%) para reducir valores atípicos en la variable `price` por grupo de barrio.

4. **Codificación de variables categóricas**
   - Se usó **One-Hot Encoding** sobre `property_type`, `room_type` y `neighbourhood_cleansed`.

5. **Normalización de variables numéricas**
   - Se aplicó **MinMaxScaler** para escalar `price`, `minimum_nights`, `maximum_nights` y `number_of_reviews`.

6. **Exportación del dataset procesado**
   - El dataset limpio se guardó en formato comprimido en la ruta:
     ```
     data/processed/listings_clean_madrid.csv.gz
     ```
   - Este archivo está listo para ser utilizado en el notebook siguiente de **feature engineering** o **modelado predictivo**.

7. **Commit al repositorio**
   - Se subieron el notebook y el dataset limpio a la rama `5-data-cleaning-and-preprocessing` del repositorio de la organización.
   - Commit realizado con el mensaje:
     ```
     feat(preprocessing): finalize preprocessing with encoding and normalization for Madrid dataset
     ```

---

✅ **Resultado final:**  
Dataset limpio, normalizado y codificado, listo para las siguientes fases del proyecto de análisis y modelado.
